In [5]:
import pandas as pd

df = pd.read_excel("C:/Users/Ceee/Desktop/sem 4/Reproducible Research/project/data/ESM.xlsx")


In [6]:
print(df.columns.tolist())


['data.point.number', 'replay id', 'replay title', 'map', 'date', 'result', 'team name', 'opposing team name', 'player name', 'car id', 'car name', 'score', 'goals', 'assists', 'saves', 'shots', 'shots conceded', 'goals conceded', 'goals conceded while last defender', 'shooting percentage', 'bpm', 'avg boost amount', 'amount collected', 'amount collected big pads', 'amount collected small pads', 'count collected big pads', 'count collected small pads', 'amount stolen', 'amount stolen big pads', 'amount stolen small pads', 'count stolen big pads', 'count stolen small pads', '0 boost time', '100 boost time', 'amount used while supersonic', 'amount overfill total', 'amount overfill stolen', 'avg speed', 'total distance', 'time slow speed', 'percentage slow speed', 'time boost speed', 'percentage boost speed', 'time supersonic speed', 'percentage supersonic speed', 'time on ground', 'percentage on ground', 'time low in air', 'percentage low in air', 'time high in air', 'percentage high in 

In [7]:
import pandas as pd

# =========================
# 1. LOAD DATA
# =========================
#df = pd.read_excel("your_file.xlsx")

print("Original shape:", df.shape)


# =========================
# 2. CLEAN COLUMN NAMES
# =========================
df.columns = (
    df.columns.str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_")
)


# =========================
# 3. RECREATE MATCH STRUCTURE
# =========================
# dataset has 2 rows per match → rebuild match_id
df = df.reset_index(drop=True)
df["match_id"] = df.index // 2


# =========================
# 4. CREATE TARGET VARIABLE
# =========================
df["goal_difference"] = df["goals"] - df["goals_conceded"]


# =========================
# 5. CREATE PLAYER INDEX
# =========================
df["player_index"] = df.groupby("match_id").cumcount()


# =========================
# 6. PIVOT (1 ROW PER MATCH)
# =========================
df_pivot = df.set_index(["match_id", "player_index"]).unstack()

# flatten column names
df_pivot.columns = [
    f"{col[0]}_{col[1]}" for col in df_pivot.columns
]

df_pivot = df_pivot.reset_index()

print("After pivot shape:", df_pivot.shape)


# =========================
# 7. CREATE DIFFERENCE FEATURES
# =========================
df_pivot["goal_diff"] = df_pivot["goals_0"] - df_pivot["goals_1"]
df_pivot["shots_diff"] = df_pivot["shots_0"] - df_pivot["shots_1"]
df_pivot["saves_diff"] = df_pivot["saves_0"] - df_pivot["saves_1"]
df_pivot["shots_conceded_diff"] = (
    df_pivot["shots_conceded_0"] - df_pivot["shots_conceded_1"]
)

# proxy for "goalside of ball"
df_pivot["goalside_diff"] = (
    df_pivot["time_behind_ball_0"] - df_pivot["time_behind_ball_1"]
)


# =========================
# 8. SAVE CLEANED DATA
# =========================
df_pivot.to_csv("cleaned_data.csv", index=False)

print("✅ Cleaned dataset saved as cleaned_data.csv")

Original shape: (8222, 76)
After pivot shape: (4111, 155)
✅ Cleaned dataset saved as cleaned_data.csv
